# Constrained creative writing in English — Kaggle runner (2× T4)WritingPrompts + four verifiable constraints, comparing plain generation againstorthogonal constraint steering.Every story is checkpointed as it is written. If this notebook stops, re-run thesame cell and it continues where it left off.Accelerator must be **GPU T4 x2**.

In [ ]:
MODEL    = "Qwen3-8B"          # Qwen3-8B | Llama-3.1-8B | Mistral-Nemo-12BSUITE    = "compare"           # compare = baseline vs our method (start here)                               # noise   = the four steering variants                               # core    = + baseline and L-Res references                               # ortho | beta | loo | allPROMPTS  = 10                  # WritingPrompts promptsSTORIES  = 5                   # stories per prompt (diversity is measured within a prompt)OUT      = "/kaggle/working/english"REPO     = "/kaggle/working/NoiseEGRA"BRANCH   = "english-generalization"

## 1. Install and clone

In [ ]:
!pip install -q -U "transformers>=4.56" accelerate hf_transfer datasets spacy!python -m spacy download -q en_core_web_sm!rm -rf {REPO} && git clone -q --branch {BRANCH} --single-branch \    https://github.com/haziq-exe/NoiseEGRA.git {REPO}import osos.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"   # HF's default downloader stalls on large shards!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 2. Hugging Face loginQwen3-8B is Apache 2.0 and needs no token. Run this only for Llama-3.1-8B, which isgated: accept the licence on its model page, then add your token under**Add-ons → Secrets** as `HF_TOKEN`.

In [ ]:
from kaggle_secrets import UserSecretsClientfrom huggingface_hub import loginlogin(UserSecretsClient().get_secret("HF_TOKEN"))

## 3. Continuing from an earlier session (skip on the first run)Kaggle wipes `/kaggle/working` when a session ends. **Save Version** before closing,then next session add that version's output under **Add-ons → Add data → Your Work**and put its path here.

In [ ]:
import os, shutil, jsonPREVIOUS = ""   # e.g. "/kaggle/input/english-run-1/english"if PREVIOUS and os.path.isdir(PREVIOUS):    shutil.copytree(PREVIOUS, OUT, dirs_exist_ok=True)    sp = f"{OUT}/{MODEL}/state.json"    n = sum(len(v) for v in json.load(open(sp))["runs"].values()) if os.path.isfile(sp) else 0    print(f"restored {n} stories")else:    print("starting fresh")

## 4. GenerateThe first run downloads the model (~16 GB) and the WritingPrompts dataset, thenextracts the steering vectors once. After that it prints one line per story with arunning estimate of time remaining. Safe to interrupt.

In [ ]:
!cd {REPO} && python -u scripts/run_english_experiment.py \    --model {MODEL} --suite {SUITE} \    --num-prompts {PROMPTS} --stories-per-prompt {STORIES} \    --out {OUT}

## 5. ScoreConstraint adherence uses the exact checks. `--diversity` adds Vendi and Self-BLEU,computed within each prompt group and then averaged.

In [ ]:
!cd {REPO} && python -u scripts/score_english.py \    --input-dir {OUT}/{MODEL} --diversity

## 6. Read some storiesNumbers cannot tell you the prose is any good. Read a few before trusting the table.

In [ ]:
import csv, glob, textwrapfor path in sorted(glob.glob(f"{OUT}/{MODEL}/*.csv")):    rows = list(csv.DictReader(open(path, encoding="utf-8")))    print("=" * 100)    print(path.split("/")[-1])    print("=" * 100)    for r in rows[:2]:        print(f"\n-- prompt {r['prompt_index']}, story {r['story_index']} --")        print(textwrap.fill(r["story"], 96))    print()